# Bronze Layer

## Objective

The Bronze Layer stores raw datasets exactly as received from the source system.

No cleaning, filtering, transformation, or business logic is applied at this stage.

This layer acts as the immutable source for downstream processing.

### Input

- learners.csv
- courses.csv
- enrolment_activity.csv

### Output

- bronze_learners
- bronze_courses
- bronze_enrolments

In [0]:
from pyspark.sql.functions import *

In [0]:
BASE_PATH = "/Volumes/workspace/default/lms_data"

LEARNERS_PATH = f"{BASE_PATH}/learners.csv"
COURSES_PATH = f"{BASE_PATH}/courses.csv"
ENROLMENT_PATH = f"{BASE_PATH}/enrolment_activity.csv"

In [0]:
bronze_learners = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(LEARNERS_PATH)
)

bronze_courses = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(COURSES_PATH)
)

bronze_enrolments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ENROLMENT_PATH)
)

In [0]:
print("Bronze Learners :", bronze_learners.count())
print("Bronze Courses :", bronze_courses.count())
print("Bronze Enrolments :", bronze_enrolments.count())

Bronze Learners : 500
Bronze Courses : 60
Bronze Enrolments : 2000


In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS lms_bronze;

In [0]:
bronze_learners.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("lms_bronze.bronze_learners")

In [0]:
bronze_courses.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("lms_bronze.bronze_courses")

In [0]:
bronze_enrolments.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("lms_bronze.bronze_enrolments")

In [0]:
%sql

SHOW TABLES IN lms_bronze;

database,tableName,isTemporary
lms_bronze,bronze_courses,false
lms_bronze,bronze_enrolments,false
lms_bronze,bronze_learners,false


In [0]:
%sql

SELECT *

FROM lms_bronze.bronze_learners

LIMIT 10;

learner_id,learner_name,email,phone_number,city,registration_date,subscription_type
LRN0001,Ananya Sharma,ananya.sharma.1@gmail.com,9433218196,Mumbai,2022-04-06,Free
LRN0002,Sachin Pillai,sachin.pillai.2@gmail.com,9083863794,Indore,2022-06-13,Premium
LRN0003,Suresh Mishra,suresh.mishra.3@rediffmail.com,9235116155,Chennai,2022-02-14,Premium
LRN0004,Vijay Kumar,vijay.kumar.4@gmail.com,9618495931,Surat,2022-08-22,Premium
LRN0005,Priya Chatterjee,priya.chatterjee.5@yahoo.com,9316475255,Surat,2022-10-01,Premium
LRN0006,Karan Pillai,karan.pillai.6@outlook.com,9283276483,Bhopal,2022-02-27,Free
LRN0007,Aditya Bose,aditya.bose.7@hotmail.com,9564139537,Surat,2023-04-15,Free
LRN0008,Divya Yadav,divya.yadav.8@yahoo.com,9884969653,Jaipur,2023-05-21,Free
LRN0009,Rohan Chatterjee,rohan.chatterjee.9@hotmail.com,9122691669,Jaipur,2022-09-15,Enterprise
LRN0010,Aarav Desai,aarav.desai.10@gmail.com,9184514627,Chandigarh,2022-09-27,Enterprise


In [0]:
%sql

SELECT *

FROM lms_bronze.bronze_courses

LIMIT 10;

course_id,course_title,category,instructor_id,instructor_name,duration_hours,difficulty_level,price_inr
CRS001,Python for Data Science,Data Science,INS009,Suresh Gupta,15,Intermediate,2986
CRS002,Machine Learning Fundamentals,AI & ML,INS006,Priya Singh,10,Beginner,223
CRS003,Deep Learning with TensorFlow,AI & ML,INS004,Sunita Rao,10,Advanced,13612
CRS004,React.js Complete Guide,Web Development,INS014,Pooja Mishra,30,Advanced,6867
CRS005,Node.js Backend Development,Web Development,INS007,Vikas Mehta,20,Beginner,119
CRS006,HTML & CSS Mastery,Web Development,INS015,null,5,Intermediate,2276
CRS007,Business Analytics Essentials,Business,INS003,Arjun Patel,20,Advanced,10604
CRS008,Project Management Professional,Business,INS003,Arjun Patel,40,Intermediate,4049
CRS009,Digital Marketing Strategy,Business,INS004,Sunita Rao,10,Advanced,8712
CRS010,UI/UX Design Principles,Design,INS005,Deepak Joshi,10,Beginner,133


In [0]:
%sql

SELECT *

FROM lms_bronze.bronze_enrolments

LIMIT 10;

enrolment_id,learner_id,course_id,enrol_date,expected_completion_date,actual_completion_date,status,progress_pct,last_activity_date,assessment_score,attempts,feedback_rating,certificate_issued
ENR00460,LRN0376,CRS001,2024-01-13,2024-02-12,2024-02-21,Completed,100,2024-02-21,79.22,1,2,Yes
ENR00681,LRN0486,CRS055,2024-01-16,2024-01-26,2024-01-19,Completed,100,2024-01-19,56.62,1,3,No
ENR01477,LRN0352,CRS044,2024-03-16,2024-06-04,2024-03-31,Completed,100,2024-03-31,93.98,1,4,Yes
ENR00554,LRN0474,CRS035,2024-02-01,2024-04-21,null,Dropped,8,2024-02-07,null,1,null,No
ENR00640,LRN0240,CRS038,2024-02-11,2024-03-12,2024-03-01,Completed,100,2024-03-01,81.21,2,3,Yes
ENR00309,LRN0457,CRS012,2024-03-09,2024-04-18,null,Dropped,28,2024-03-10,null,1,null,No
ENR00406,LRN0316,CRS060,2024-02-22,2024-03-23,2024-03-27,Completed,100,2024-03-27,96.0,1,4,Yes
ENR00933,LRN0309,CRS042,2024-03-06,2024-03-16,null,Dropped,11,2024-03-09,null,1,null,No
ENR01153,LRN0078,CRS012,2024-02-06,2024-03-17,null,In Progress,15,2024-03-27,null,1,null,No
ENR00810,LRN0424,CRS039,2024-01-30,2024-04-19,null,Dropped,35,2024-02-10,null,2,null,No
